In [ ]:
!pip install xgboost -q

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/man-bot-detection'
DATA_BASE = os.path.join(BASE_DIR, 'data/raw')

PATHS = {
    'CIC': os.path.join(DATA_BASE, 'cic_ids_2017'),
    'UNSW': os.path.join(DATA_BASE, 'unsw_nb15'),
    'CTU': os.path.join(DATA_BASE, 'ctu13')
}

OUTPUT_DIR = os.path.join(BASE_DIR, 'notebooks/results')
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)


def load_cic():
    path = PATHS['CIC']
    all_files = glob.glob(os.path.join(path, '*.csv'))
    if not all_files:
        raise FileNotFoundError(path)
    df_list = [pd.read_csv(f, low_memory=False) for f in all_files]
    df = pd.concat(df_list, ignore_index=True)
    if len(df) > 200000:
        df = df.sample(n=200000, random_state=42)
    print(len(df), len(df.columns))
    return df

def load_unsw():
    path = PATHS['UNSW']
    train_file = os.path.join(path, 'UNSW_NB15_training-set.csv')
    test_file = os.path.join(path, 'UNSW_NB15_testing-set.csv')
    if not os.path.exists(train_file) or not os.path.exists(test_file):
        raise FileNotFoundError(path)
    df_train = pd.read_csv(train_file, low_memory=False)
    df_test = pd.read_csv(test_file, low_memory=False)
    df = pd.concat([df_train, df_test], ignore_index=True)
    if len(df) > 200000:
        df = df.sample(n=200000, random_state=42)
    print(len(df), len(df.columns))
    return df

def load_ctu():
    path = PATHS['CTU']
    attack_file = os.path.join(path, 'CTU13_Attack_Traffic.csv')
    normal_file = os.path.join(path, 'CTU13_Normal_Traffic.csv')
    if not os.path.exists(attack_file) or not os.path.exists(normal_file):
        raise FileNotFoundError(path)
    df_attack = pd.read_csv(attack_file, low_memory=False)
    df_normal = pd.read_csv(normal_file, low_memory=False)
    raw_label_variants = ['Label', 'LABEL', 'Label_original']
    df_attack = df_attack.drop(columns=[c for c in raw_label_variants if c in df_attack.columns],
                                errors='ignore')
    df_normal = df_normal.drop(columns=[c for c in raw_label_variants if c in df_normal.columns],
                                errors='ignore')
    common_cols = df_attack.columns.intersection(df_normal.columns)
    dropped_attack = set(df_attack.columns) - set(common_cols)
    dropped_normal = set(df_normal.columns) - set(common_cols)
    if dropped_attack or dropped_normal:
        print(sorted(dropped_attack), sorted(dropped_normal))
    df_attack = df_attack[common_cols]
    df_normal = df_normal[common_cols]

    df_attack['label'] = 1
    df_normal['label'] = 0
    df = pd.concat([df_attack, df_normal], ignore_index=True)
    print(len(df), len(df.columns), int((df['label']==1).sum()), int((df['label']==0).sum()))
    return df

def detect_label_column(df):
    candidates = ['label', 'Label', 'Label_original', 'Class', 'attack_cat']
    for col in candidates:
        if col in df.columns:
            return col
    for col in df.columns:
        if 'label' in col.lower() or 'class' in col.lower():
            return col
    raise ValueError(df.columns.tolist())


def preprocess_data(df, label_col):
    df = df.replace([np.inf, -np.inf], np.nan).dropna()

    X = df.drop(columns=[label_col])
    y = df[label_col]

    if y.dtype == 'object':
        y = y.astype(str).str.lower()
        y = y.map(lambda x: 0 if x in ['benign', 'normal', '0'] else 1)
    else:
        y = y.astype(int)

    if len(np.unique(y)) < 2:
        print(np.unique(y, return_counts=True))
        for col in df.columns:
            if 'label' in col.lower() or 'attack' in col.lower():
                alt_y = df[col]
                if alt_y.dtype == 'object':
                    alt_y = alt_y.astype(str).str.lower()
                    alt_y = alt_y.map(lambda x: 0 if x in ['normal', 'benign', '0'] else 1)
                else:
                    alt_y = alt_y.astype(int)
                if len(np.unique(alt_y)) == 2:
                    print(col)
                    y = alt_y
                    break
        if len(np.unique(y)) < 2:
            raise ValueError(y)

    print(sum(y==0), sum(y==1))

    drop_keywords = ['attack_cat', 'Label_original', 'Class']
    cols_to_drop = []
    for col in X.columns:
        if col in drop_keywords:
            cols_to_drop.append(col)
    X = X.drop(columns=cols_to_drop, errors='ignore')

    num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    print(len(num_cols), len(cat_cols))

    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    transformers = []
    if num_cols:
        transformers.append(('num', StandardScaler(), num_cols))
    if cat_cols:
        transformers.append(('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols))

    preprocessor = ColumnTransformer(transformers, remainder='drop')
    X_processed = preprocessor.fit_transform(X)

    feature_names = []
    if num_cols:
        feature_names.extend(num_cols)
    if cat_cols:
        cat_encoder = preprocessor.named_transformers_['cat']
        feature_names.extend(cat_encoder.get_feature_names_out(cat_cols))

    X_processed = pd.DataFrame(X_processed, columns=feature_names)

    print(len(feature_names))
    print(feature_names[:10])

    return X_processed, y, preprocessor, X


def train_and_evaluate(X, y, model_name, model):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return {
        'model': model_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='binary', zero_division=0),
        'recall': recall_score(y_test, y_pred, average='binary', zero_division=0),
        'f1': f1_score(y_test, y_pred, average='binary', zero_division=0)
    }


results = []
raw_X_clean = {}
raw_X_processed = {}
raw_y = {}

loaders = {
    'CIC': load_cic,
    'UNSW': load_unsw,
    'CTU': load_ctu
}

for name, loader in loaders.items():
    print(name)
    try:
        df = loader()
        label_col = detect_label_column(df)
        print(label_col)

        X_processed, y, preprocessor, X_clean = preprocess_data(df, label_col)

        raw_X_clean[name] = X_clean
        raw_X_processed[name] = X_processed
        raw_y[name] = y

        models = {
            'RandomForest': RandomForestClassifier(n_estimators=50, random_state=42, class_weight='balanced', n_jobs=-1),
            'XGBoost': XGBClassifier(n_estimators=50, random_state=42, use_label_encoder=False, eval_metric='logloss',
                                     tree_method='hist', scale_pos_weight=sum(y==0)/sum(y==1))
        }

        for model_name, model in models.items():
            print(model_name)
            res = train_and_evaluate(X_processed, y, model_name, model)
            res['dataset'] = name
            results.append(res)

            if model_name == 'RandomForest':
                importances = model.feature_importances_
                indices = np.argsort(importances)[::-1]
                for i in range(min(10, len(indices))):
                    print(X_processed.columns[indices[i]], importances[indices[i]])

            print(res['accuracy'], res['f1'])

    except Exception as e:
        print(e)


FEATURE_MAP_CIC_CTU = {
    'flow_duration':        {'CIC': ' Flow Duration',               'CTU': 'Flow Duration'},
    'total_fwd_packets':     {'CIC': ' Total Fwd Packets',           'CTU': 'Tot Fwd Pkts'},
    'total_bwd_packets':     {'CIC': ' Total Backward Packets',      'CTU': 'Tot Bwd Pkts'},
    'total_fwd_bytes':       {'CIC': 'Total Length of Fwd Packets',  'CTU': 'TotLen Fwd Pkts'},
    'total_bwd_bytes':       {'CIC': ' Total Length of Bwd Packets', 'CTU': 'TotLen Bwd Pkts'},
    'fwd_pkt_len_max':       {'CIC': ' Fwd Packet Length Max',       'CTU': 'Fwd Pkt Len Max'},
    'fwd_pkt_len_min':       {'CIC': ' Fwd Packet Length Min',       'CTU': 'Fwd Pkt Len Min'},
    'fwd_pkt_len_mean':      {'CIC': ' Fwd Packet Length Mean',      'CTU': 'Fwd Pkt Len Mean'},
    'fwd_pkt_len_std':       {'CIC': ' Fwd Packet Length Std',       'CTU': 'Fwd Pkt Len Std'},
    'bwd_pkt_len_max':       {'CIC': 'Bwd Packet Length Max',        'CTU': 'Bwd Pkt Len Max'},
    'bwd_pkt_len_min':       {'CIC': ' Bwd Packet Length Min',       'CTU': 'Bwd Pkt Len Min'},
    'bwd_pkt_len_mean':      {'CIC': ' Bwd Packet Length Mean',      'CTU': 'Bwd Pkt Len Mean'},
    'bwd_pkt_len_std':       {'CIC': ' Bwd Packet Length Std',       'CTU': 'Bwd Pkt Len Std'},
    'flow_bytes_per_s':      {'CIC': 'Flow Bytes/s',                 'CTU': 'Flow Byts/s'},
    'flow_packets_per_s':    {'CIC': ' Flow Packets/s',              'CTU': 'Flow Pkts/s'},
    'flow_iat_mean':         {'CIC': ' Flow IAT Mean',               'CTU': 'Flow IAT Mean'},
    'flow_iat_std':          {'CIC': ' Flow IAT Std',                'CTU': 'Flow IAT Std'},
    'flow_iat_max':          {'CIC': ' Flow IAT Max',                'CTU': 'Flow IAT Max'},
    'flow_iat_min':          {'CIC': ' Flow IAT Min',                'CTU': 'Flow IAT Min'},
    'fwd_iat_total':         {'CIC': 'Fwd IAT Total',                'CTU': 'Fwd IAT Tot'},
    'fwd_iat_mean':          {'CIC': ' Fwd IAT Mean',                'CTU': 'Fwd IAT Mean'},
    'fwd_iat_std':           {'CIC': ' Fwd IAT Std',                 'CTU': 'Fwd IAT Std'},
    'fwd_iat_max':           {'CIC': ' Fwd IAT Max',                 'CTU': 'Fwd IAT Max'},
    'fwd_iat_min':           {'CIC': ' Fwd IAT Min',                 'CTU': 'Fwd IAT Min'},
    'bwd_iat_total':         {'CIC': 'Bwd IAT Total',                'CTU': 'Bwd IAT Tot'},
    'bwd_iat_mean':          {'CIC': ' Bwd IAT Mean',                'CTU': 'Bwd IAT Mean'},
    'bwd_iat_std':           {'CIC': ' Bwd IAT Std',                 'CTU': 'Bwd IAT Std'},
    'bwd_iat_max':           {'CIC': ' Bwd IAT Max',                 'CTU': 'Bwd IAT Max'},
    'bwd_iat_min':           {'CIC': ' Bwd IAT Min',                 'CTU': 'Bwd IAT Min'},
    'bwd_psh_flags':         {'CIC': ' Bwd PSH Flags',               'CTU': 'Bwd PSH Flags'},
    'fwd_header_len':        {'CIC': ' Fwd Header Length',           'CTU': 'Fwd Header Len'},
    'bwd_header_len':        {'CIC': ' Bwd Header Length',           'CTU': 'Bwd Header Len'},
    'fwd_packets_per_s':     {'CIC': 'Fwd Packets/s',                'CTU': 'Fwd Pkts/s'},
    'bwd_packets_per_s':     {'CIC': ' Bwd Packets/s',               'CTU': 'Bwd Pkts/s'},
    'min_packet_len':        {'CIC': ' Min Packet Length',           'CTU': 'Pkt Len Min'},
    'max_packet_len':        {'CIC': ' Max Packet Length',           'CTU': 'Pkt Len Max'},
    'packet_len_mean':       {'CIC': ' Packet Length Mean',          'CTU': 'Pkt Len Mean'},
    'packet_len_std':        {'CIC': ' Packet Length Std',           'CTU': 'Pkt Len Std'},
    'packet_len_var':        {'CIC': ' Packet Length Variance',      'CTU': 'Pkt Len Var'},
    'fin_flag_count':        {'CIC': 'FIN Flag Count',               'CTU': 'FIN Flag Cnt'},
    'syn_flag_count':        {'CIC': ' SYN Flag Count',              'CTU': 'SYN Flag Cnt'},
    'rst_flag_count':        {'CIC': ' RST Flag Count',              'CTU': 'RST Flag Cnt'},
    'ack_flag_count':        {'CIC': ' ACK Flag Count',              'CTU': 'ACK Flag Cnt'},
    'down_up_ratio':         {'CIC': ' Down/Up Ratio',               'CTU': 'Down/Up Ratio'},
    'avg_packet_size':       {'CIC': ' Average Packet Size',         'CTU': 'Pkt Size Avg'},
    'avg_fwd_segment_size':  {'CIC': ' Avg Fwd Segment Size',        'CTU': 'Fwd Seg Size Avg'},
    'avg_bwd_segment_size':  {'CIC': ' Avg Bwd Segment Size',        'CTU': 'Bwd Seg Size Avg'},
    'init_win_bytes_bwd':    {'CIC': ' Init_Win_bytes_backward',     'CTU': 'Init Bwd Win Byts'},
    'act_data_pkt_fwd':      {'CIC': ' act_data_pkt_fwd',            'CTU': 'Fwd Act Data Pkts'},
    'active_mean':           {'CIC': 'Active Mean',                  'CTU': 'Active Mean'},
    'active_std':            {'CIC': ' Active Std',                  'CTU': 'Active Std'},
    'active_max':            {'CIC': ' Active Max',                  'CTU': 'Active Max'},
    'active_min':            {'CIC': ' Active Min',                  'CTU': 'Active Min'},
    'idle_mean':             {'CIC': 'Idle Mean',                    'CTU': 'Idle Mean'},
    'idle_std':              {'CIC': ' Idle Std',                    'CTU': 'Idle Std'},
    'idle_max':              {'CIC': ' Idle Max',                    'CTU': 'Idle Max'},
    'idle_min':              {'CIC': ' Idle Min',                    'CTU': 'Idle Min'},
}

FEATURE_MAP_CIC_UNSW = {
    'flow_duration':      {'CIC': ' Flow Duration',              'UNSW': 'dur'},
    'total_fwd_packets':  {'CIC': ' Total Fwd Packets',          'UNSW': 'spkts'},
    'total_bwd_packets':  {'CIC': ' Total Backward Packets',     'UNSW': 'dpkts'},
    'total_fwd_bytes':    {'CIC': 'Total Length of Fwd Packets', 'UNSW': 'sbytes'},
    'total_bwd_bytes':    {'CIC': ' Total Length of Bwd Packets','UNSW': 'dbytes'},
    'fwd_pkt_len_mean':   {'CIC': ' Fwd Packet Length Mean',     'UNSW': 'smean'},
    'bwd_pkt_len_mean':   {'CIC': ' Bwd Packet Length Mean',     'UNSW': 'dmean'},
    'flow_packets_per_s': {'CIC': ' Flow Packets/s',             'UNSW': 'rate'},
}


def build_common_features(feature_map, datasets):
    names = list(datasets.keys())
    resolved = {name: [] for name in names}
    used_keys = []
    for key, mapping in feature_map.items():
        ok = all(mapping.get(name) in datasets[name].columns for name in names)
        if ok:
            used_keys.append(key)
            for name in names:
                resolved[name].append(mapping[name])
        else:
            missing = {name: mapping.get(name) for name in names
                       if mapping.get(name) not in datasets[name].columns}
            print(key, missing)

    print(len(used_keys), used_keys)

    out = {}
    for name in names:
        out[name] = datasets[name][resolved[name]].copy()
        out[name].columns = used_keys
    return out, used_keys


def train_with_optimal_threshold_and_test(feature_map, train_name, train_df, train_y,
                                           target_name, target_df, target_y, pair_label):
    print(pair_label)

    aligned, used_keys = build_common_features(
        feature_map, {train_name: train_df, target_name: target_df}
    )
    if len(used_keys) == 0:
        print(pair_label)
        return []

    X_train_full = aligned[train_name]
    X_target = aligned[target_name]

    scaler = StandardScaler()
    X_train_full_scaled = scaler.fit_transform(X_train_full)
    X_target_scaled = scaler.transform(X_target)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_full_scaled, train_y, test_size=0.2, random_state=42, stratify=train_y
    )

    models_cross = {
        'RandomForest': RandomForestClassifier(n_estimators=50, random_state=42,
                                                class_weight='balanced', n_jobs=-1),
        'XGBoost': XGBClassifier(n_estimators=50, random_state=42, use_label_encoder=False,
                                  eval_metric='logloss', tree_method='hist',
                                  scale_pos_weight=sum(y_tr == 0) / max(sum(y_tr == 1), 1)),
    }

    results_pair = []
    for model_name, model in models_cross.items():
        model.fit(X_tr, y_tr)

        proba_val = model.predict_proba(X_val)[:, 1]
        thresholds = np.linspace(0, 1, 101)
        f1_scores = [f1_score(y_val, (proba_val >= th).astype(int), zero_division=0)
                     for th in thresholds]
        best_idx = int(np.argmax(f1_scores))
        best_th = thresholds[best_idx]
        print(model_name, best_th, f1_scores[best_idx])

        proba_target = model.predict_proba(X_target_scaled)[:, 1]

        print(proba_val.mean(), np.median(proba_val), np.percentile(proba_val, 90))
        print(proba_target.mean(), np.median(proba_target), np.percentile(proba_target, 90))
        n_above_th = int((proba_target >= best_th).sum())
        print(n_above_th, len(proba_target), 100*n_above_th/len(proba_target))

        y_pred = (proba_target >= best_th).astype(int)

        acc = accuracy_score(target_y, y_pred)
        prec = precision_score(target_y, y_pred, zero_division=0)
        rec = recall_score(target_y, y_pred, zero_division=0)
        f1 = f1_score(target_y, y_pred, zero_division=0)

        results_pair.append({
            'train': train_name, 'test': target_name, 'model': model_name,
            'n_features': len(used_keys), 'threshold': best_th,
            'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
        })
        print(model_name, train_name, target_name, len(used_keys), f1, acc, prec, rec)

    return results_pair


cross_results = []
cross_results += train_with_optimal_threshold_and_test(
    FEATURE_MAP_CIC_CTU, 'CIC', raw_X_clean['CIC'], raw_y['CIC'],
    'CTU', raw_X_clean['CTU'], raw_y['CTU'], 'CIC -> CTU'
)
cross_results += train_with_optimal_threshold_and_test(
    FEATURE_MAP_CIC_UNSW, 'CIC', raw_X_clean['CIC'], raw_y['CIC'],
    'UNSW', raw_X_clean['UNSW'], raw_y['UNSW'], 'CIC -> UNSW'
)


df_results = pd.DataFrame(results)
if not df_results.empty:
    pivot_table = df_results.pivot_table(index='dataset', columns='model',
                                         values=['accuracy', 'precision', 'recall', 'f1']).round(4)
    print(pivot_table.to_string())
    pivot_table.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'internal_comparison.csv'))

df_cross = pd.DataFrame(cross_results)
if not df_cross.empty:
    cross_pivot = df_cross.pivot_table(index='test', columns='model', values=['f1', 'accuracy']).round(4)
    print(cross_pivot.to_string())
    df_cross.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'cross_with_threshold.csv'), index=False)

    plt.figure(figsize=(10, 6))
    ax = sns.barplot(data=df_cross, x='model', y='f1', hue='test')

    max_f1 = df_cross['f1'].max()
    y_top = max(max_f1 * 1.8, 0.02)
    plt.ylim(0, y_top)

    if not df_results.empty:
        for model_name, color in [('RandomForest', 'tab:blue'), ('XGBoost', 'tab:orange')]:
            internal_f1 = df_results.loc[df_results['model'] == model_name, 'f1'].mean()
            if pd.notna(internal_f1):
                plt.axhline(y=min(internal_f1, y_top), color=color, linestyle=':',
                             linewidth=1.5, alpha=0.6)

    plt.title('F1')
    plt.ylabel('F1')
    plt.xlabel('model')
    plt.grid(axis='y', alpha=0.3)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.4f', padding=3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'cross_f1_threshold.png'), dpi=300)
    plt.show()

print(OUTPUT_DIR)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os

OUTPUT_DIR = os.path.join(BASE_DIR, 'notebooks/results')
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_results, x='dataset', y='accuracy', hue='model')
plt.title('accuracy')
plt.ylim(0, 1.05)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'accuracy_comparison.png'), dpi=300)
plt.show()

if 'df_cv' in globals() and not df_cv.empty:
    plt.figure(figsize=(10, 6))
    ax = sns.barplot(data=df_cv, x='Dataset', y='Mean F1', hue='Model')
    for i, row in df_cv.iterrows():
        plt.errorbar(x=i, y=row['Mean F1'], yerr=row['Std F1'],
                     fmt='none', capsize=5, color='black', elinewidth=2)
    plt.title('F1')
    plt.ylim(0, 1.05)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'cv_f1.png'), dpi=300)
    plt.show()
else:
    print('df_cv')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, name in enumerate(['CIC', 'UNSW', 'CTU']):
    y = raw_y[name]
    counts = pd.Series(y).value_counts().sort_index()
    axes[idx].bar(counts.index, counts.values, color=['blue', 'orange'])
    axes[idx].set_title(name)
    axes[idx].set_xlabel('class')
    axes[idx].set_ylabel('count')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'class_distribution.png'), dpi=300)
plt.show()

print(os.path.join(OUTPUT_DIR, 'figures', 'accuracy_comparison.png'))
print(os.path.join(OUTPUT_DIR, 'figures', 'cv_f1.png'))
print(os.path.join(OUTPUT_DIR, 'figures', 'class_distribution.png'))

if 'FEATURE_MAP_CIC_CTU' in globals():
    print(len(FEATURE_MAP_CIC_CTU))
if 'FEATURE_MAP_CIC_UNSW' in globals():
    print(len(FEATURE_MAP_CIC_UNSW))

for name in ['CIC', 'UNSW', 'CTU']:
    X = raw_X_clean[name]
    print(name, X.shape[1], list(X.columns[:10]))

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv_results = []
for name in ['CIC', 'UNSW', 'CTU']:
    X = raw_X_processed[name]
    y = raw_y[name]

    non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric:
        print(name, non_numeric)
        X = pd.get_dummies(X, columns=non_numeric, dummy_na=True)

    print(name, X.shape, y.shape)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for model_name in ['RandomForest', 'XGBoost']:
        if model_name == 'RandomForest':
            model = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
        else:
            model = XGBClassifier(n_estimators=50, random_state=42, use_label_encoder=False, eval_metric='logloss')

        scores = cross_val_score(model, X, y, cv=skf, scoring='f1', n_jobs=-1)
        mean_f1 = np.mean(scores)
        std_f1 = np.std(scores)
        cv_results.append({
            'Dataset': name,
            'Model': model_name,
            'Mean F1': mean_f1,
            'Std F1': std_f1
        })
        print(model_name, mean_f1, std_f1)

df_cv = pd.DataFrame(cv_results)
print(df_cv)
df_cv.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'cv_metrics.csv'), index=False)